# SlideRule: on-demand ICESat-2 and GEDI processing

[SlideRule](https://slideruleearth.io/) processes satellite altimetry and
lidar data in the cloud. `ea.plugins.sliderule` sends a processor, its
parameters and an area of interest, and returns geolocated points as a
`GeoDataFrame`. SlideRule manages its own credentials.

## Setup

```bash
pip install earthaccess geopandas matplotlib panel geoviews datashader
pip install "earthaccess-plugin-sliderule @ git+https://github.com/betolink/earthaccess-plugin-sliderule"
```

Register at https://slideruleearth.io and use your organization below.

In [ ]:
import earthaccess as ea

ea.login()
ea.plugins.sliderule.configure(url="slideruleearth.io", organization="sliderule")
ea.list_plugins(details=True)

## Search

A small ATL03 (ICESat-2) region over Greenland.

In [ ]:
bbox = (-43.5, 66.5, -43.3, 66.7)  # a small ATL03 region

atl03 = ea.search_data(
    short_name="ATL03",
    bounding_box=bbox,
    temporal=("2023-06-01", "2023-06-30"),
    count=1,
)
len(atl03)

## What SlideRule can do

The plugin reports the processors it can run and the subsetting it does:

In [ ]:
ea.plugins.sliderule.capabilities("ATL03")

## Process with `atl03x`

`request(...).results()` sends the request and returns a `GeoDataFrame`. The searched
granules scope the request; the `aoi` restricts the region.

In [ ]:
aoi = {
    "type": "Polygon",
    "coordinates": [
        [[-43.5, 66.5], [-43.3, 66.5], [-43.3, 66.7], [-43.5, 66.7], [-43.5, 66.5]]
    ],
}

df = ea.plugins.sliderule.request(atl03, api="atl03x", parms={}, aoi=aoi).results()
df.shape

In [ ]:
df.head()

In [ ]:
df.plot(column="height", markersize=0.5)  # geolocated points from SlideRule

## Photon returns: along-track profile and plan view

`atl03x` can also classify every photon with the ATL08 algorithm. Passing
`atl08_class` adds the return type (`0` noise, `1` ground, `2` canopy,
`3` top of canopy, `4` unclassified), enough to look at vegetation structure
along a beam.

We pull two ATL03 granules over Boulder, Colorado and cache the returned
`GeoDataFrame` as GeoParquet, so re-running the notebook reuses the local file
instead of re-submitting the SlideRule request.

In [ ]:
from pathlib import Path

import geopandas as gpd

# Boulder, Colorado
boulder_bbox = (-105.35, 39.95, -105.15, 40.10)  # west, south, east, north
boulder_aoi = {
    "type": "Polygon",
    "coordinates": [
        [
            [-105.35, 39.95],
            [-105.15, 39.95],
            [-105.15, 40.10],
            [-105.35, 40.10],
            [-105.35, 39.95],
        ]
    ],
}

cache = Path("boulder_photons.parquet")

if cache.exists():
    photons = gpd.read_parquet(cache)  # GeoParquet cache
else:
    atl03_boulder = ea.search_data(
        short_name="ATL03",
        bounding_box=boulder_bbox,
        temporal=("2023-06-01", "2023-08-31"),
        count=2,
    )
    parms_atl08 = {
        "cnf": -2,  # keep every confidence level, including noise
        "atl08_class": [
            "atl08_noise",
            "atl08_ground",
            "atl08_canopy",
            "atl08_top_of_canopy",
            "atl08_unclassified",
        ],
    }
    photons = ea.plugins.sliderule.request(
        atl03_boulder, api="atl03x", parms=parms_atl08, aoi=boulder_aoi
    ).results()
    photons.to_parquet(cache)  # GeoParquet cache for the next run

photons.shape

The returned `GeoDataFrame` carries `x_atc` (along-track distance), `height`,
`atl08_class` and `atl03_cnf`. On the left, the along-track profile colored by
return type; on the right, the same photons seen from above.

Both panes are a single **categorical datashader aggregation**
(`datashader.by("return_type", count())`), so all categories are rendered in one
pass and the view stays interactive at hundreds of thousands of points. The
slider resamples the `GeoDataFrame`; both panes are rebuilt from the same sample,
so they stay in sync.

In [ ]:
import cartopy.crs as ccrs
import datashader as ds
import geoviews as gv
import holoviews as hv
import panel as pn
from holoviews.operation.datashader import dynspread, rasterize

gv.extension("bokeh")
pn.extension()

return_types = {
    0: "noise",
    1: "ground",
    2: "canopy",
    3: "top of canopy",
    4: "unclassified",
}
colors = {
    "noise": "#bdbdbd",
    "ground": "#7f3b08",
    "canopy": "#006d2c",
    "top of canopy": "#66c2a4",
    "unclassified": "#fdd49e",
}

returns = photons.assign(return_type=photons["atl08_class"].map(return_types))
returns = returns.dropna(subset=["return_type"])
returns["longitude"] = returns.geometry.x
returns["latitude"] = returns.geometry.y


def photon_panes(percent):
    """Rasterize a random `percent` sample by return type into two synced panes."""
    sample = returns.sample(frac=percent / 100, random_state=0)
    aggregate = dict(aggregator=ds.by("return_type", ds.count()), color_key=colors)

    along_track = dynspread(
        rasterize(hv.Points(sample, ["x_atc", "height"], "return_type"), **aggregate)
    ).opts(
        title="Along-track photon profile",
        width=560,
        height=460,
        xlabel="Along-track distance, x_atc (m)",
        ylabel="Height (m)",
    )
    plan_view = (
        gv.tile_sources.ESRI
        * dynspread(
            rasterize(
                gv.Points(
                    sample,
                    ["longitude", "latitude"],
                    "return_type",
                    crs=ccrs.PlateCarree(),
                ),
                **aggregate,
            )
        )
    ).opts(title="Plan view", width=560, height=460, projection=ccrs.PlateCarree())

    return pn.Row(along_track, plan_view)


sample_slider = pn.widgets.DiscreteSlider(
    name="Sample (%)", options=[1, 2, 5, 10, 25, 50, 100], value=10
)

pn.Column(sample_slider, pn.bind(photon_panes, sample_slider))

## Other processors and parameters

Pass any SlideRule API and its parameters. ICESat-2 APIs include `atl03sp`,
`atl06p`, `atl06sp`, `atl08p`, `atl13sp`, `atl24v`; GEDI APIs include
`gedil1b`, `gedil2a`, `gedil4a`. `parms` is passed through unchanged:

```python
segments = ea.plugins.sliderule.request(
    atl03, api="atl06p", parms={"pass_invalid": False}, aoi=aoi,
).results()
```

You can also run from an `aoi` + `parms` alone, with no search results.

In [ ]:
segments = ea.plugins.sliderule.request(
    atl03, api="atl06p", parms={}, aoi=aoi
).results()
segments.shape

## Unsupported datasets fail early

SlideRule only handles ICESat-2 and GEDI; anything else raises before the
backend is contacted:

In [ ]:
from earthaccess.plugins import UnsupportedCollectionError

mursst = ea.search_data(short_name="MUR-JPL-L4-GLOB-v4.1", count=1)
try:
    ea.plugins.sliderule.request(mursst, api="atl06p").results()
except UnsupportedCollectionError as exc:
    print(exc)

## Which plugins accept a dataset?

In [ ]:
atl03.list_services()

## Summary

- `ea.plugins.sliderule.request(results, api=..., parms=..., aoi=...).results()`
  → `GeoDataFrame`.
- SlideRule keeps its own credentials (`configure(...)`).
- Supports only ICESat-2 and GEDI collections; others fail early.